<a href="https://colab.research.google.com/github/Shreyans06/LLMs-understanding/blob/main/Self-Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [4]:
# Generating context vector using Weight matrices(Query , Key , Value)
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [5]:
# Initialize weight matrices
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in , d_out) , requires_grad = False)
W_key = torch.nn.Parameter(torch.rand(d_in , d_out) , requires_grad = False)
W_value = torch.nn.Parameter(torch.rand(d_in , d_out) , requires_grad = False)

In [6]:
query2 = x_2 @ W_query
key2 = x_2 @ W_key
value2 = x_2 @ W_value
print(query2)

tensor([0.4306, 1.4551])


In [7]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:" , keys.shape)
print("values.shape:" , values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [8]:
keys_2 = keys[1]
attn_score_22 = query2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [9]:
attn_scores_2 = query2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [10]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5 , dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [11]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


Self-Attention class

In [12]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
  def __init__(self, d_in , d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in , d_out))
    self.W_key = nn.Parameter(torch.rand(d_in , d_out))
    self.W_value = nn.Parameter(torch.rand(d_in , d_out))

  def forward(self , x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5 , dim=-1)
    context_vec = attn_weights @ values
    return context_vec


In [13]:
torch.manual_seed(123)
self_attn = SelfAttention_v1(d_in = d_in , d_out = d_out)
print(self_attn(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Self Attention using Linear Layers

In [14]:
class SelfAttention_v2(nn.Module):
  def __init__(self , d_in , d_out , qkv_bias = False):
    super().__init__()
    self.W_query = nn.Linear(d_in , d_out , bias = qkv_bias)
    self.W_key = nn.Linear(d_in , d_out , bias = qkv_bias)
    self.W_value = nn.Linear(d_in , d_out , bias = qkv_bias)

  def forward(self , x):
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5 , dim = -1
    )
    context_vec = attn_weights @ values
    return context_vec


In [15]:
torch.manual_seed(789)
self_attn_v2 = SelfAttention_v2(d_in , d_out)
print(self_attn_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Causal attention mask

In [16]:
queries = self_attn_v2.W_query(inputs)
keys = self_attn_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1] ** 0.5 , dim=-1
)
print(attn_weights)


tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [17]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length , context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [18]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [19]:
row_sums = masked_simple.sum(dim=-1 , keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [20]:
mask = torch.triu(torch.ones(context_length , context_length) , diagonal=1)
print(mask)
print(attn_scores)
masked = attn_scores.masked_fill(mask.bool() , -torch.inf)
print(masked)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackw

In [21]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5 , dim = 1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [22]:
context_vec = attn_weights @ values
print(context_vec)

tensor([[0.1855, 0.8812],
        [0.2795, 0.9361],
        [0.3133, 0.9508],
        [0.2994, 0.8595],
        [0.2702, 0.7554],
        [0.2772, 0.7618]], grad_fn=<MmBackward0>)


In [23]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6 , 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [24]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [25]:
batch = torch.stack((inputs , inputs) , dim = 0)
print(batch.shape)

torch.Size([2, 6, 3])


In [26]:
class CausalAttention(nn.Module):
  def __init__(self , d_in , d_out , context_length , dropout , qkv_bias = False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in , d_out , bias = qkv_bias)
    self.W_key = nn.Linear(d_in , d_out , bias = qkv_bias)
    self.W_value = nn.Linear(d_in , d_out , bias = qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length , context_length) , diagonal = 1)
    )

  def forward(self , x):
    b , num_tokens , d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1 , 2)
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens , :num_tokens] , -torch.inf
    )
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5 , dim =-1
    )
    attn_weights = self.dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec

In [27]:
torch.manual_seed(123)
context_length = batch.shape[1]
causal_attn = CausalAttention(d_in , d_out , context_length , 0.0)
context_vecs = causal_attn(batch)
print("Context vectors shape" , context_vecs.shape)

Context vectors shape torch.Size([2, 6, 2])
